# 05 - Feature Engineering och preprocessing

## Syfte

Syftet med denna notebook är att skapa de features och den preprocessing
som ska användas i den senare modelleringen.

Feature engineering baseras på resultaten från EDA:n i föregående notebook.
Endast information som är tillgänglig vid predictionstillfället används,
så att target leakage undviks.

Preprocessing byggs så att alla steg som lär sig information från data
fit:as endast på träningsdata. Samma transformer används därefter
oförändrade på validation- och testdata.

Notebooken omfattar:

- val av features utifrån EDA-resultaten
- skapande av tidsbaserade och geografiska features
- hantering av saknade numeriska värden
- kodning av kategoriska variabler
- skalning av numeriska variabler
- fit av preprocessing endast på träningsdata
- transformering av train, validation och test
- sparande av preprocessing, feature-lista och färdiga feature-tabeller
  som artifacts för nästa notebook

In [36]:
from pathlib import Path

import pandas as pd
import numpy as np

# Sökväg till artifacts från tidigare notebooks
artifacts_dir = Path("artifacts")

# Läs in de tre dataset som skapades i split-notebooken
train = pd.read_csv(artifacts_dir / "train.csv")
validation = pd.read_csv(artifacts_dir / "validation.csv")
test = pd.read_csv(artifacts_dir / "test.csv")

# Kontrollera storlek och att target finns i alla dataset
print("Train shape:", train.shape)
print("Validation shape:", validation.shape)
print("Test shape:", test.shape)

print("\nTarget finns:")
print("Train:", "late" in train.columns)
print("Validation:", "late" in validation.columns)
print("Test:", "late" in test.columns)

Train shape: (67533, 21)
Validation shape: (14471, 21)
Test shape: (14472, 21)

Target finns:
Train: True
Validation: True
Test: True


## Predictionstillfälle och data leakage

Modellen ska uppskatta risken för att en order blir sen utifrån information
som är tillgänglig tidigt i orderprocessen.

Features som beskriver själva ordern, betalningen, kundens och säljarens
geografiska information samt den beräknade leveranstiden kan därför
användas.

Information som uppstår senare i leveransprocessen får däremot inte användas.
Exempelvis exkluderas faktisk leveranstid, leveransdatum till kund och
transportörens leveransdatum eftersom dessa variabler innehåller information
om framtiden i förhållande till predictionstillfället.

Identifierare och variabler med mycket hög cardinality används inte heller
som direkta modellfeatures.

## Skapande av features

Utifrån resultaten från EDA:n skapas nya features från information som är
tillgänglig vid predictionstillfället.

Från `order_purchase_timestamp` skapas månad, veckodag och timme.
Den planerade leveranstiden beräknas från orderdatum och beräknat
leveransdatum.

Dessutom skapas `same_state`, som anger om kunden och säljaren finns i
samma state. EDA:n visade att denna information har ett tydligt samband
med risken för sen leverans.

Samma feature engineering appliceras på train-, validation- och testdata.

In [37]:
# Skapa samma features för train, validation och test
def create_features(df):
    df = df.copy()

    # Konvertera datum som behövs för feature engineering
    df["order_purchase_timestamp"] = pd.to_datetime(
        df["order_purchase_timestamp"]
    )
    df["order_estimated_delivery_date"] = pd.to_datetime(
        df["order_estimated_delivery_date"]
    )

    # Tidsfeatures från ordertillfället
    df["purchase_month"] = df["order_purchase_timestamp"].dt.month
    df["purchase_weekday"] = df["order_purchase_timestamp"].dt.weekday
    df["purchase_hour"] = df["order_purchase_timestamp"].dt.hour

    # Planerad leveranstid, känd vid ordertillfället
    df["estimated_delivery_days"] = (
        df["order_estimated_delivery_date"]
        - df["order_purchase_timestamp"]
    ).dt.total_seconds() / 86400

    # Geografisk feature
    df["same_state"] = (
        df["customer_state"] == df["seller_state"]
    ).astype(int)

    return df


train_fe = create_features(train)
validation_fe = create_features(validation)
test_fe = create_features(test)

new_features = [
    "purchase_month",
    "purchase_weekday",
    "purchase_hour",
    "estimated_delivery_days",
    "same_state"
]

print(train_fe[new_features].head())
print("\nMissing values i nya features:")
print(train_fe[new_features].isna().sum())

   purchase_month  purchase_weekday  purchase_hour  estimated_delivery_days  \
0               9                 3             12                18.488449   
1              10                 0              9                23.593866   
2              10                 0             16                34.293866   
3              10                 0             21                52.123831   
4              10                 0             21                56.115556   

   same_state  
0           0  
1           0  
2           0  
3           0  
4           0  

Missing values i nya features:
purchase_month             0
purchase_weekday           0
purchase_hour              0
estimated_delivery_days    0
same_state                 0
dtype: int64


## Val av modellfeatures

Feature-urvalet baseras på resultaten från EDA:n och på vilken information
som är tillgänglig vid predictionstillfället.

Numeriska features består av information om ordern, betalningen,
leveransavståndet samt de tids- och geografiska features som skapades ovan.

`customer_state` och `seller_state` används som kategoriska features eftersom
EDA:n visade geografiska skillnader i risken för sen leverans.

Identifierare, råa postnummerprefix och information som uppstår senare i
leveransprocessen exkluderas för att undvika hög dimensionalitet och
data leakage.

In [38]:
# Numeriska features
numeric_features = [
    "total_items",
    "total_price",
    "total_freight",
    "total_payment_value",
    "number_of_payments",
    "max_installments",
    "seller_count",
    "distance_km",
    "purchase_month",
    "purchase_weekday",
    "purchase_hour",
    "estimated_delivery_days",
    "same_state"
]

# Kategoriska features
categorical_features = [
    "customer_state",
    "seller_state"
]

feature_columns = numeric_features + categorical_features

print("Antal numeriska features:", len(numeric_features))
print("Antal kategoriska features:", len(categorical_features))
print("Totalt antal features före encoding:", len(feature_columns))

print("\nNumeriska features:")
print(numeric_features)

print("\nKategoriska features:")
print(categorical_features)

Antal numeriska features: 13
Antal kategoriska features: 2
Totalt antal features före encoding: 15

Numeriska features:
['total_items', 'total_price', 'total_freight', 'total_payment_value', 'number_of_payments', 'max_installments', 'seller_count', 'distance_km', 'purchase_month', 'purchase_weekday', 'purchase_hour', 'estimated_delivery_days', 'same_state']

Kategoriska features:
['customer_state', 'seller_state']


In [39]:
# Separera features och target
X_train = train_fe[feature_columns].copy()
X_validation = validation_fe[feature_columns].copy()
X_test = test_fe[feature_columns].copy()

y_train = train_fe["late"].copy()
y_validation = validation_fe["late"].copy()
y_test = test_fe["late"].copy()

# Kontrollera dimensioner
print("X_train:", X_train.shape)
print("X_validation:", X_validation.shape)
print("X_test:", X_test.shape)

print("\ny_train:", y_train.shape)
print("y_validation:", y_validation.shape)
print("y_test:", y_test.shape)

# Kontrollera saknade värden före preprocessing
missing_train = X_train.isna().sum()
missing_train = missing_train[missing_train > 0].sort_values(ascending=False)

print("\nSaknade värden i X_train före preprocessing:")
print(missing_train)

X_train: (67533, 15)
X_validation: (14471, 15)
X_test: (14472, 15)

y_train: (67533,)
y_validation: (14471,)
y_test: (14472,)

Saknade värden i X_train före preprocessing:
distance_km            344
total_payment_value      1
number_of_payments       1
max_installments         1
dtype: int64


## Preprocessing

De numeriska variablerna innehåller ett mindre antal saknade värden.
Dessa hanteras med medianimputering, vilket är lämpligt eftersom flera
numeriska variabler är högerskeva och innehåller extrema observationer.

Numeriska features standardiseras med `StandardScaler`.

De kategoriska variablerna `customer_state` och `seller_state` imputeras
vid behov med det vanligaste värdet och kodas med `OneHotEncoder`.
Okända kategorier tillåts vid transformering av validation- och testdata.

Alla preprocessing-steg fit:as endast på träningsdata. Validation- och
testdata transformeras därefter med exakt samma fit:ade preprocessing
för att undvika data leakage.

In [40]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# Preprocessing för numeriska features
numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]
)

# Preprocessing för kategoriska features
categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        (
            "onehot",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False
            )
        )
    ]
)

# Kombinera numerisk och kategorisk preprocessing
preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", numeric_transformer, numeric_features),
        ("categorical", categorical_transformer, categorical_features)
    ]
)

print(preprocessor)

ColumnTransformer(transformers=[('numeric',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='median')),
                                                 ('scaler', StandardScaler())]),
                                 ['total_items', 'total_price', 'total_freight',
                                  'total_payment_value', 'number_of_payments',
                                  'max_installments', 'seller_count',
                                  'distance_km', 'purchase_month',
                                  'purchase_weekday', 'purchase_hour',
                                  'estimated_delivery_days', 'same_state']),
                                ('categorical',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='most_frequent')),
                                                 ('onehot',
                       

In [41]:
# Fit preprocessing endast på träningsdata
X_train_processed = preprocessor.fit_transform(X_train)

# Applicera samma fit:ade preprocessing på validation och test
X_validation_processed = preprocessor.transform(X_validation)
X_test_processed = preprocessor.transform(X_test)

print("Processed train shape:", X_train_processed.shape)
print("Processed validation shape:", X_validation_processed.shape)
print("Processed test shape:", X_test_processed.shape)

print("\nSaknade värden efter preprocessing:")
print("Train:", np.isnan(X_train_processed).sum())
print("Validation:", np.isnan(X_validation_processed).sum())
print("Test:", np.isnan(X_test_processed).sum())

Processed train shape: (67533, 62)
Processed validation shape: (14471, 62)
Processed test shape: (14472, 62)

Saknade värden efter preprocessing:
Train: 0
Validation: 0
Test: 0


In [42]:
# Hämta slutliga feature-namn efter preprocessing
processed_feature_names = preprocessor.get_feature_names_out()

# Skapa DataFrames med tydliga kolumnnamn
X_train_processed_df = pd.DataFrame(
    X_train_processed,
    columns=processed_feature_names,
    index=X_train.index
)

X_validation_processed_df = pd.DataFrame(
    X_validation_processed,
    columns=processed_feature_names,
    index=X_validation.index
)

X_test_processed_df = pd.DataFrame(
    X_test_processed,
    columns=processed_feature_names,
    index=X_test.index
)

print("Antal slutliga features:", len(processed_feature_names))
print("\nFörsta 10 feature-namn:")
print(processed_feature_names[:10])

print("\nShapes:")
print("Train:", X_train_processed_df.shape)
print("Validation:", X_validation_processed_df.shape)
print("Test:", X_test_processed_df.shape)

Antal slutliga features: 62

Första 10 feature-namn:
['numeric__total_items' 'numeric__total_price' 'numeric__total_freight'
 'numeric__total_payment_value' 'numeric__number_of_payments'
 'numeric__max_installments' 'numeric__seller_count'
 'numeric__distance_km' 'numeric__purchase_month'
 'numeric__purchase_weekday']

Shapes:
Train: (67533, 62)
Validation: (14471, 62)
Test: (14472, 62)


In [43]:
# Skapa slutliga modelldataset med target som sista kolumn
train_model = X_train_processed_df.copy()
validation_model = X_validation_processed_df.copy()
test_model = X_test_processed_df.copy()

train_model["late"] = y_train.to_numpy()
validation_model["late"] = y_validation.to_numpy()
test_model["late"] = y_test.to_numpy()

# Kontrollera slutliga dataset
print("Train model shape:", train_model.shape)
print("Validation model shape:", validation_model.shape)
print("Test model shape:", test_model.shape)

print("\nTargetfördelning:")
print("Train:")
print(train_model["late"].value_counts().sort_index())

print("\nValidation:")
print(validation_model["late"].value_counts().sort_index())

print("\nTest:")
print(test_model["late"].value_counts().sort_index())

print("\nSaknade värden:")
print("Train:", train_model.isna().sum().sum())
print("Validation:", validation_model.isna().sum().sum())
print("Test:", test_model.isna().sum().sum())

Train model shape: (67533, 63)
Validation model shape: (14471, 63)
Test model shape: (14472, 63)

Targetfördelning:
Train:
late
0    61436
1     6097
Name: count, dtype: int64

Validation:
late
0    13698
1      773
Name: count, dtype: int64

Test:
late
0    13515
1      957
Name: count, dtype: int64

Saknade värden:
Train: 0
Validation: 0
Test: 0


In [44]:
import json
import joblib

# Spara färdiga modelldataset
train_model.to_csv(artifacts_dir / "model_train.csv", index=False)
validation_model.to_csv(artifacts_dir / "model_validation.csv", index=False)
test_model.to_csv(artifacts_dir / "model_test.csv", index=False)

# Spara fit:ad preprocessing
joblib.dump(preprocessor, artifacts_dir / "preprocessor.joblib")

# Spara slutlig feature-lista
feature_list = processed_feature_names.tolist()

with open(artifacts_dir / "feature_list.json", "w", encoding="utf-8") as f:
    json.dump(feature_list, f, ensure_ascii=False, indent=2)

print("Sparade artifacts:")
print("-", artifacts_dir / "model_train.csv")
print("-", artifacts_dir / "model_validation.csv")
print("-", artifacts_dir / "model_test.csv")
print("-", artifacts_dir / "preprocessor.joblib")
print("-", artifacts_dir / "feature_list.json")

Sparade artifacts:
- artifacts\model_train.csv
- artifacts\model_validation.csv
- artifacts\model_test.csv
- artifacts\preprocessor.joblib
- artifacts\feature_list.json


In [45]:
# Slutlig kontroll av artifacts från feature engineering
artifact_files = [
    "model_train.csv",
    "model_validation.csv",
    "model_test.csv",
    "preprocessor.joblib",
    "feature_list.json"
]

print("Kontroll av artifacts:\n")

for filename in artifact_files:
    path = artifacts_dir / filename
    print(f"{filename}: {'OK' if path.exists() else 'SAKNAS'}")

print("\nSammanfattning:")
print("Antal features före encoding:", len(feature_columns))
print("Antal features efter preprocessing:", len(processed_feature_names))
print("Train rows:", len(train_model))
print("Validation rows:", len(validation_model))
print("Test rows:", len(test_model))
print("Missing values efter preprocessing:",
      train_model.isna().sum().sum()
      + validation_model.isna().sum().sum()
      + test_model.isna().sum().sum())

Kontroll av artifacts:

model_train.csv: OK
model_validation.csv: OK
model_test.csv: OK
preprocessor.joblib: OK
feature_list.json: OK

Sammanfattning:
Antal features före encoding: 15
Antal features efter preprocessing: 62
Train rows: 67533
Validation rows: 14471
Test rows: 14472
Missing values efter preprocessing: 0


## RESULTAT


I denna notebook skapades de features och den preprocessing som ska användas
i modelleringen.

Feature engineering baserades på resultaten från EDA:n och begränsades till
information som är tillgänglig vid predictionstillfället. Nya features
skapades från ordertid, beräknad leveranstid och geografisk information.

Totalt valdes 15 features före preprocessing:
13 numeriska och 2 kategoriska.

Numeriska saknade värden hanterades med medianimputering och de numeriska
variablerna standardiserades med `StandardScaler`. Kategoriska variabler
kodades med `OneHotEncoder`, med stöd för kategorier som inte förekommer
i träningsdata.

All preprocessing fit:ades endast på träningsdata. Samma fit:ade transformer
användes därefter oförändrad på validation- och testdata för att undvika
data leakage.

Efter preprocessing skapades 62 slutliga modellfeatures. Samtliga tre
dataset innehåller samma feature-struktur och inga saknade värden.

Följande artifacts sparades för nästa notebook:

- `artifacts/model_train.csv`
- `artifacts/model_validation.csv`
- `artifacts/model_test.csv`
- `artifacts/preprocessor.joblib`
- `artifacts/feature_list.json`

`preprocessor.joblib` innehåller den fit:ade imputeringen, skalningen och
kategorikodningen och kan därför återanvändas konsekvent.